<a href="https://colab.research.google.com/github/Lorenzito/ProgramacionParaAnaliticaDescriptivayPredicitva/blob/main/Sesion12_Evaluacion_Datos_Categoricos_273524.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: Lorenzo Varela Ollervides
- **Matrícula** 273524

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [105]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [106]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [107]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.

for col in df.columns:
  if df[col].dtype == 'object':    # Filtramos para solo mostrar las columas de tipo object
    print(df[col].value_counts())  # Imprimimos para cada columna que cumple la condición los valores dentro de ellas y cuantas veces se repiten
    print()




customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64

gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection
No                     3095
Yes      

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_ Si, la columna "TotalCharges" esta clasificada como de tipo object ya que aunque la mayoria de los valores son float64, hay 11 valores que se muestran vacios. No se ven valores que difieran por mayúsculas o espacios.

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [108]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [109]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)

pd.crosstab(df['OnlineSecurity'], df['InternetService'])




InternetService,DSL,Fiber optic,No
OnlineSecurity,,,
No,1241,2257,0
No internet service,0,0,1526
Yes,1180,839,0


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ No, no es un valor inválido. Es una columna legítima ya que al hacer el crosstab entre las columnas 'OnlineSecurity' e 'InternetService' podemos observar que justamente los registros que no cuentan con internet son los que tienen esa categoria seleccionada. La conservaría tal cual.

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [110]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset (no solo las categóricas)
# y la razón (valores únicos / total de filas) para cada una.

for col in df.columns:
    unicos = df[col].nunique()
    razon = unicos / df.shape[0]
    print(f"{col:<20}{'Valores únicos:':<18}{unicos:<10}{'Razón:':<10}{razon:.4f}")


customerID          Valores únicos:   7043      Razón:    1.0000
gender              Valores únicos:   2         Razón:    0.0003
SeniorCitizen       Valores únicos:   2         Razón:    0.0003
Partner             Valores únicos:   2         Razón:    0.0003
Dependents          Valores únicos:   2         Razón:    0.0003
tenure              Valores únicos:   73        Razón:    0.0104
PhoneService        Valores únicos:   2         Razón:    0.0003
MultipleLines       Valores únicos:   3         Razón:    0.0004
InternetService     Valores únicos:   3         Razón:    0.0004
OnlineSecurity      Valores únicos:   3         Razón:    0.0004
OnlineBackup        Valores únicos:   3         Razón:    0.0004
DeviceProtection    Valores únicos:   3         Razón:    0.0004
TechSupport         Valores únicos:   3         Razón:    0.0004
StreamingTV         Valores únicos:   3         Razón:    0.0004
StreamingMovies     Valores únicos:   3         Razón:    0.0004
Contract            Valor

**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ CustomerID, TotalCharges son las variables con una cardinalidad mayor al 90%. Para la columna con mayor cardinalidad "CustomerID" nunca deberia de usarse como variable predictora en un modelo porque puede generar problemas de sobreajuste si los modelos aprenden patrones irrelevantes ademas de aumentar el costo computacional y dificultar la interpretación de los resultados. Esta columna fue pensada como un identificador único por lo que es esperado que todos los valores presentan en ella sean diferentes.

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [111]:
# Aplica el agrupamiento Top 10 + Otros sobre la columna de mayor cardinalidad

top10 = df['customerID'].value_counts().head(10)
categorias_top10 = top10.index.tolist()

df['customerID_agrupado'] = df['customerID'].where(
    df['customerID'].isin(categorias_top10), 'Otros')

df['customerID_agrupado'].value_counts()



,count
customerID_agrupado,
Otros,7033
7590-VHVEG,1
5575-GNVDE,1
9837-FWLCH,1
1699-HPSBG,1
7203-OYKCT,1
1035-IPQPU,1
7398-LXGYX,1
2823-LKABH,1


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_ No me parece útil, la columna con mayor cardinalidad en este dataset es un ID de cliente, es decir un identificador único, agruparlos es absurdo considerando que se crearon justamante para ser valores irrepetibles. A comparación de 'country' es que una variable donde tenemos una cantidad finita de posibles respuestas lo que nos permite agruparlas ya que unos pocos valores concentran gran parte de las filas.

---
## Actividad 4 — Tipos de dato (30 pts)

In [112]:
df['TotalCharges'].dtype

dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_ Despues de investigar pude encontrar los valores responsables de que Pandas lo clasifique como tipo object en lugar de float64, algunos valores tenian " " un string vacio que no permitio el cast directo a Object.

In [113]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificamos que solo sean NaN las filas que detectamos antes y que el tipo ya sea float64
print("NaN tras la conversión:", df['TotalCharges'].isna().sum())
print("Tipo:", df['TotalCharges'].dtype)

# Los clientes con NaN tienen tenure = 0, o sea que aún no se les ha facturado nada,
# por lo que su cargo total es 0. Imputamos 0 en lugar de eliminar las filas
# para no perder la información de esos clientes.
df.loc[df['TotalCharges'].isna(), 'TotalCharges'] = 0

# Comprobamos que ya no queden valores nulos
print("NaN finales:", df['TotalCharges'].isna().sum())
df['TotalCharges'].dtype


NaN tras la conversión: 11
Tipo: float64
NaN finales: 0


dtype('float64')

In [114]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.

# Excluimos customerID porque es un identificador único (razón = 1.0), no una categoría real;
# convertirla a category no aporta nada y solo ocuparía memoria de más.
excluir = ['customerID']

for col in df.columns:
    if col in excluir:
        continue
    unicos = df[col].nunique()
    razon = unicos / len(df)
    # Criterio basado en la Actividad 3: menos de 10 valores únicos y razón menor a 0.01,
    # es decir, pocos valores distintos que se repiten muchas veces (valores fijos).
    # Con esto, tenure (73 únicos), MonthlyCharges y TotalCharges (numéricas continuas)
    # se quedan sin convertir. SeniorCitizen sí se convierte, porque aunque es numérica (0/1)
    # solo tiene 2 valores fijos que representan una categoría.
    if unicos < 10 and razon < 0.01:
        df[col] = df[col].astype('category')

df.dtypes

,0
customerID,object
gender,category
SeniorCitizen,category
Partner,category
Dependents,category
tenure,int64
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_
1. Los valores inválidos fueron los que me pareceron más difíciles de decidir ya que necesitamos tener un contexto general del dataset para poder tomas la decisión, en este caso, fue necesario hacer un crosstab con otra columna para definir que 'NoInternetService' era un valor legítimo. La más fácil para mi fue la cardinalidad ya que vasta con conocer el total de valore únicos en la columna y dividirlo entre el total de registros.
2. Le diría que CustomerID es un identificador único por lo que no vale la pena intentar agruparlo, no debe usarse como variable predictora de un modelo porque puede generar sobreajuste si es un patron irrelevante. Y sobre TotalCharges le diría que sufrio una transformación convirtiendo los valores a float64 ya que inicialmente Pandas las clasificaba como object al tener string vacios.

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.